## 1. Setup

In [ ]:
# 3_kaggle_12691.94159.ipynb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Sklearn Tools and Preprocessing Kits
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Models
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import ElasticNet, Lasso, Ridge

## 2. DataSet

In [34]:
# Local
# base_path = (
#     r"C:\Users\User\Documents\GitHub\template-kaggle-competition\housing-prices\data"
# )
# X_full = pd.read_csv(f"{base_path}\\train.csv", index_col="Id")
# X_test_full = pd.read_csv(f"{base_path}\\test.csv", index_col="Id")

# kaggle
X_full = pd.read_csv('/kaggle/input/competitions/home-data-for-ml-course/train.csv', index_col="Id")
X_test_full = pd.read_csv('/kaggle/input/competitions/home-data-for-ml-course/test.csv', index_col="Id")

# Remove extreme outliers
outliers = X_full[(X_full['GrLivArea'] > 4000) & (X_full['SalePrice'] < 300000)].index
X_full.drop(outliers, axis=0, inplace=True)

# Remove data without target value
X_full.dropna(axis=0, subset=["SalePrice"], inplace=True)

# Objective variable logarithmic transformation
y_log = np.log1p(X_full.SalePrice)
X_full.drop(["SalePrice"], axis=1, inplace=True)

# For fields with missing values ​​that have actual physical meaning, fill in 'None' or 0
none_cols = [
    "PoolQC",
    "MiscFeature",
    "Alley",
    "Fence",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
]

for col in none_cols:
    if col in X_full.columns:
        X_full[col] = X_full[col].fillna("None")
        X_test_full[col] = X_test_full[col].fillna("None")

zero_cols = [
    "GarageArea",
    "GarageCars",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "BsmtFullBath",
    "BsmtHalfBath",
]
for col in zero_cols:
    if col in X_full.columns:
        X_full[col] = X_full[col].fillna(0)
        X_test_full[col] = X_test_full[col].fillna(0)

# Displays the number of data entries and the number of columns
print(f"Training set shape: {X_full.shape}")
print(f"Test set shape: {X_test_full.shape}")

Training set shape: (1458, 79)
Test set shape: (1459, 79)


## 3. Feature Engineering

In [35]:
def add_custom_features(df):
    df_out = df.copy()
    if "MSSubClass" in df_out.columns:
        df_out["MSSubClass"] = df_out["MSSubClass"].astype(str)

    # Space and Area
    df_out["TotalSF"] = df_out["TotalBsmtSF"] + df_out["1stFlrSF"] + df_out["2ndFlrSF"]
    df_out["TotalBaths"] = (
        df_out["FullBath"]
        + (0.5 * df_out["HalfBath"])
        + df_out["BsmtFullBath"]
        + (0.5 * df_out["BsmtHalfBath"])
    )
    df_out["TotalOutsideSF"] = (
        df_out["WoodDeckSF"]
        + df_out["OpenPorchSF"]
        + df_out["EnclosedPorch"]
        + df_out["3SsnPorch"]
        + df_out["ScreenPorch"]
    )

    # Time and building age
    df_out["HouseAge"] = df_out["YrSold"] - df_out["YearBuilt"]
    df_out["RemodAge"] = df_out["YrSold"] - df_out["YearRemodAdd"]
    df_out["IsRemodeled"] = (df_out["YearBuilt"] != df_out["YearRemodAdd"]).astype(int)

    # Proportion and State
    df_out["Pct_LivArea_Total"] = df_out["GrLivArea"] / (df_out["TotalSF"] + 1e-5)
    df_out["HasBasement"] = (df_out["TotalBsmtSF"] > 0).astype(int)
    df_out["HasGarage"] = (df_out["GarageArea"] > 0).astype(int)

    # Interactive features 互動
    df_out["OverallQual_GrLivArea"] = df_out["OverallQual"] * df_out["GrLivArea"]
    df_out["OverallQual_TotalSF"] = df_out["OverallQual"] * df_out["TotalSF"]
    df_out["GarageScore"] = df_out["GarageCars"] * df_out["GarageArea"]
    df_out["OverallCond_HouseAge"] = df_out["OverallCond"] * df_out["HouseAge"]

    return df_out


X_full_fe = add_custom_features(X_full)
X_test_full_fe = add_custom_features(X_test_full)

## 4. Feature clustering (numerical and categorical) and coding definition

In [36]:
ordinal_cols = [
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "HeatingQC",
    "KitchenQual",
    "FireplaceQu",
    "GarageQual",
    "GarageCond",
]
qual_order = ["None", "Po", "Fa", "TA", "Gd", "Ex"]
ordinal_categories = [qual_order for _ in ordinal_cols]

categorical_cols = [
    cname
    for cname in X_full_fe.columns
    if X_full_fe[cname].dtype == "object" and cname not in ordinal_cols
]
numerical_cols = [
    cname
    for cname in X_full_fe.columns
    if X_full_fe[cname].dtype in ["int64", "float64"]
]

my_cols = numerical_cols + ordinal_cols + categorical_cols
X = X_full_fe[my_cols].copy()
X_test = X_test_full_fe.reindex(columns=X.columns)

## 5. Data preprocessing pipeline

In [37]:
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import skew


# Declare a bias converter to prevent data leakage 宣告偏態轉換器
class SkewnessTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.75):
        self.threshold = threshold
        self.skewed_idx_ = []

    def fit(self, X, y=None):
        # exclude_feats = [
        #     "OverallQual_GrLivArea",
        #     "OverallQual_TotalSF",
        #     "GarageScore",
        #     "OverallCond_HouseAge",
        #     "IsRemodeled",
        #     "HasBasement",
        #     "HasGarage",
        # ]
        X_df = pd.DataFrame(X)
        self.skewed_idx_ = [
            i for i in X_df.columns
            if abs(skew(X_df[i].dropna())) > self.threshold
        ]
        return self

    def transform(self, X):
        X_out = np.array(X, dtype=float, copy=True)
        for i in self.skewed_idx_:
            X_out[:, i] = np.log1p(np.maximum(0, X_out[:, i]))
        return X_out


# -- 1. tree models specific pipeline (declared independently, never sharing variables with linear models 獨立宣告，不與線模型共用變數) --
ord_transformer_tree = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "ordinal",
            OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

cat_transformer_tree = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

num_transformer_tree = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("skew_corr", SkewnessTransformer(threshold=0.75)),
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", num_transformer_tree, numerical_cols),
        ("ord", ord_transformer_tree, ordinal_cols),
        ("cat", cat_transformer_tree, categorical_cols),
    ]
)


# -- 2. Linear model specific pipeline (completely copied to prevent StandardScaler scaling crashes caused by shared objects 新複製，防止共用物件造成 StandardScaler 縮放崩潰) --
ord_transformer_linear = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "ordinal",
            OrdinalEncoder(
                categories=ordinal_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

cat_transformer_linear = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

num_transformer_linear = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("skew_corr", SkewnessTransformer(threshold=0.75)),
        ("scaler", StandardScaler()),  # Standardized scaling for linear models
    ]
)

preprocessor_linear = ColumnTransformer(
    transformers=[
        ("num", num_transformer_linear, numerical_cols),
        ("ord", ord_transformer_linear, ordinal_cols),
        ("cat", cat_transformer_linear, categorical_cols),
    ]
)

## 6. Global Validation Setup and RMSLE Evaluation Tools

In [38]:
from sklearn.metrics import root_mean_squared_log_error


class Config:
    N_SPLITS = 10
    RANDOM_STATE = 42
    SHUFFLE = True
    XGB_PARAMS = {
        "n_estimators": 1500,
        "learning_rate": 0.05,
        "max_depth": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "early_stopping_rounds": 100,
        "eval_metric": "rmse",
        "random_state": 42,
        "n_jobs": -1,
    }

    # LightGBM Hyperparameters
    LGB_PARAMS = {
        "n_estimators": 1500,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }

    # CatBoost Hyperparameters
    CAT_PARAMS = {
        "iterations": 1500,
        "learning_rate": 0.05,
        "depth": 6,
        "random_state": 42,
        "verbose": False,
    }


def evaluate_rmsle(y_true_log, y_pred_log):
    mse = mean_squared_error(y_true_log, y_pred_log)
    return np.sqrt(mse)

## 7. Establish a group of linear models (Lasso / ElasticNet / Ridge) OOF

In [39]:
from sklearn.linear_model import Lasso, ElasticNet, Ridge
from sklearn.model_selection import KFold


def run_linear_oof_training(X, y_log, X_test, preprocessor):
    models = {
        "Lasso": Lasso(alpha=0.0005, max_iter=10000, random_state=42),
        "ElasticNet": ElasticNet(
            alpha=0.0005, l1_ratio=0.5, max_iter=10000, random_state=42
        ),
        "Ridge": Ridge(alpha=12),
    }
    oof_preds = {name: np.zeros(len(X)) for name in models}
    test_preds = {name: np.zeros(len(X_test)) for name in models}
    kf = KFold(
        n_splits=Config.N_SPLITS,
        shuffle=Config.SHUFFLE,
        random_state=Config.RANDOM_STATE,
    )

    print(f"Start of linear model group {Config.N_SPLITS}-Fold OOF cross-validation")
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
        X_train, y_train = X.iloc[train_idx], y_log.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y_log.iloc[val_idx]

        X_train_trans = preprocessor.fit_transform(X_train)
        X_val_trans = preprocessor.transform(X_val)
        X_test_trans = preprocessor.transform(X_test)

        for name, model in models.items():
            model.fit(X_train_trans, y_train)
            oof_preds[name][val_idx] = model.predict(X_val_trans)
            test_preds[name] += model.predict(X_test_trans) / Config.N_SPLITS

    for name in models:
        score = evaluate_rmsle(y_log, oof_preds[name])
        print(f"{name} Overall OOF RMSLE: {score:.5f}")
    return oof_preds, test_preds

## 8. Build the OOF function of the XGBoost model

In [40]:
# Establish a 3-tree model OOF training function
from sklearn.model_selection import KFold
import numpy as np
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor, early_stopping
from catboost import CatBoostRegressor


# -- 1. XGBoost Training function --
def run_xgb_oof_training(X, y_log, X_test, preprocessor):
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    kf = KFold(
        n_splits=Config.N_SPLITS,
        shuffle=Config.SHUFFLE,
        random_state=Config.RANDOM_STATE,
    )
    print(f"Start of XGBoost model {Config.N_SPLITS}-Fold OOF cross-validation")

    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
        X_train, y_train = X.iloc[train_idx], y_log.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y_log.iloc[val_idx]

        # Tree model preprocessing
        X_train_trans = preprocessor.fit_transform(X_train)
        X_val_trans = preprocessor.transform(X_val)
        X_test_trans = preprocessor.transform(X_test)

        model = XGBRegressor(**Config.XGB_PARAMS)
        model.fit(
            X_train_trans,
            y_train,
            eval_set=[(X_val_trans, y_val)],
            verbose=False,
        )

        oof_preds[val_idx] = model.predict(X_val_trans)
        test_preds += model.predict(X_test_trans) / Config.N_SPLITS

    print(f"XGBoost Overall OOF RMSLE: {evaluate_rmsle(y_log, oof_preds):.5f}\n")
    return oof_preds, test_preds


# -- 2. LightGBM Training function --
def run_lgb_oof_training(X, y_log, X_test, preprocessor):
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    kf = KFold(
        n_splits=Config.N_SPLITS,
        shuffle=Config.SHUFFLE,
        random_state=Config.RANDOM_STATE,
    )
    print(f"Start of LightGBM model {Config.N_SPLITS}-Fold OOF cross-validation")

    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
        X_train, y_train = X.iloc[train_idx], y_log.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y_log.iloc[val_idx]

        X_train_trans = preprocessor.fit_transform(X_train)
        X_val_trans = preprocessor.transform(X_val)
        X_test_trans = preprocessor.transform(X_test)

        model = LGBMRegressor(**Config.LGB_PARAMS)
        model.fit(
            X_train_trans,
            y_train,
            eval_set=[(X_val_trans, y_val)],
            callbacks=[
                early_stopping(stopping_rounds=100, verbose=False)
            ],  # LightGBM-specific earlystop syntax
        )

        oof_preds[val_idx] = model.predict(X_val_trans)
        test_preds += model.predict(X_test_trans) / Config.N_SPLITS

    print(f"LightGBM Overall OOF RMSLE: {evaluate_rmsle(y_log, oof_preds):.5f}\n")
    return oof_preds, test_preds


# -- 3. CatBoost Training function --
def run_cat_oof_training(X, y_log, X_test, preprocessor):
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    kf = KFold(
        n_splits=Config.N_SPLITS,
        shuffle=Config.SHUFFLE,
        random_state=Config.RANDOM_STATE,
    )
    print(f"Start of CatBoost model {Config.N_SPLITS}-Fold OOF cross-validation")

    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_log)):
        X_train, y_train = X.iloc[train_idx], y_log.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y_log.iloc[val_idx]

        X_train_trans = preprocessor.fit_transform(X_train)
        X_val_trans = preprocessor.transform(X_val)
        X_test_trans = preprocessor.transform(X_test)

        model = CatBoostRegressor(**Config.CAT_PARAMS)
        model.fit(
            X_train_trans,
            y_train,
            eval_set=[(X_val_trans, y_val)],
            early_stopping_rounds=100,  # CatBoost EarlyStop
            verbose=False,
        )

        oof_preds[val_idx] = model.predict(X_val_trans)
        test_preds += model.predict(X_test_trans) / Config.N_SPLITS

    print(f"CatBoost Overall OOF RMSLE: {evaluate_rmsle(y_log, oof_preds):.5f}\n")
    return oof_preds, test_preds

## 9. Perform training and model fusion

In [41]:
# Perform OOF training on 4 sets of models
oof_xgb, test_preds_xgb = run_xgb_oof_training(X, y_log, X_test, preprocessor_tree)
oof_lgb, test_preds_lgb = run_lgb_oof_training(X, y_log, X_test, preprocessor_tree)
oof_cat, test_preds_cat = run_cat_oof_training(X, y_log, X_test, preprocessor_tree)

oof_linears, test_preds_linears = run_linear_oof_training(
    X, y_log, X_test, preprocessor_linear
)

Start of XGBoost model 10-Fold OOF cross-validation
XGBoost Overall OOF RMSLE: 0.11556

Start of LightGBM model 10-Fold OOF cross-validation


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

LightGBM Overall OOF RMSLE: 0.12245

Start of CatBoost model 10-Fold OOF cross-validation
CatBoost Overall OOF RMSLE: 0.11285

Start of linear model group 10-Fold OOF cross-validation
Lasso Overall OOF RMSLE: 0.11001
ElasticNet Overall OOF RMSLE: 0.11085
Ridge Overall OOF RMSLE: 0.11135


In [42]:
from scipy.optimize import minimize

# Compile OOF matrix
oof_matrix = np.column_stack(
    [oof_xgb, oof_lgb, oof_cat, oof_linears["Lasso"], oof_linears["Ridge"]]
)

# Consolidate the test set prediction matrix
test_matrix = np.column_stack(
    [
        test_preds_xgb,
        test_preds_lgb,
        test_preds_cat,
        test_preds_linears["Lasso"],
        test_preds_linears["Ridge"],
    ]
)
model_names = ["XGBoost", "LightGBM", "CatBoost", "Lasso", "Ridge"]


# Optimize objective function
def loss_function(weights):
    blend_oof_preds = np.dot(oof_matrix, weights)
    return evaluate_rmsle(y_log, blend_oof_preds)


# Limitations
constraints = {"type": "eq", "fun": lambda w: 1.0 - np.sum(w)}
bounds = [(0, 1)] * len(model_names)
initial_weights = [1.0 / len(model_names)] * len(model_names)

# Find the optimal solution
optimization_result = minimize(
    loss_function,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
)
best_weights = optimization_result.x

print("\n Weight optimization completed")
for name, weight in zip(model_names, best_weights):
    print(f"{name} Optimal fusion weights: {weight:.4f}")
print(f"Blended model final (optimized) OOF RMSLE: {optimization_result.fun:.5f}\n")


 Weight optimization completed
XGBoost Optimal fusion weights: 0.2827
LightGBM Optimal fusion weights: 0.0000
CatBoost Optimal fusion weights: 0.1392
Lasso Optimal fusion weights: 0.5571
Ridge Optimal fusion weights: 0.0211
Blended model final (optimized) OOF RMSLE: 0.10701



## 10. Predict and remit

In [43]:
# Generate final test set predictions
final_test_preds_log = np.dot(test_matrix, best_weights)
final_preds_dollar = np.expm1(final_test_preds_log)

# Inspection and Export
print("Does the test set contain null values? ", np.isnan(final_preds_dollar).any())
print(f"training set average house price: {np.expm1(y_log).mean():,.0f}")  # Should ≈ 180k
# final_preds_dollar's mean should ≈ 160~180k、max < 1000k
print("Predicted values descriptive statistics:\n", pd.Series(final_preds_dollar).describe())

Does the test set contain null values?  False
training set average house price: 180,933
Predicted values descriptive statistics:
 count      1459.000000
mean     178327.540533
std       77630.197585
min       45229.135968
25%      126724.899702
50%      156513.869417
75%      210763.116134
max      800232.278440
dtype: float64


In [44]:
output = pd.DataFrame({"Id": X_test.index, "SalePrice": final_preds_dollar})
output.to_csv("submission.csv", index=False)
print(output.head())

     Id      SalePrice
0  1461  121817.763819
1  1462  161222.456985
2  1463  180800.413808
3  1464  195532.790947
4  1465  190526.727215
